# Task 0 — Data Exploratory Analysis (EDA) sul Dataset QEvasion

Questo notebook analizza il dataset utilizzato per il fine-tuning di Llama 3.1. 
L'obiettivo è estrarre statistiche utili sulle distribuzioni delle etichette (Clarity ed Evasion), calcolare la lunghezza media dei testi e visualizzare la correlazione tra le tecniche di evasione e le macro-categorie di chiarezza.

In [2]:
# ============================================================
# CELLA 1 — Setup e Importazione Librerie
# ============================================================
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_from_disk

# Stile dei grafici
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# Mount Google Drive (solo su Colab)
try:
    import google.colab
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    print("Google Drive montato con successo.")
except ImportError:
    print("Ambiente locale rilevato.")

# Percorso base per salvare i grafici dell'analisi (Opzionale)
ANALYSIS_DIR = "/content/drive/MyDrive/progettoLLM/analisi_dataset"
os.makedirs(ANALYSIS_DIR, exist_ok=True)

/home/zizzis/Code/Politecnico/Large_Language_Models/CLARITY/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Ambiente locale rilevato.


PermissionError: [Errno 13] Permission denied: '/content'

In [ ]:
# ============================================================
# CELLA 2 — Caricamento Dataset e Conversione in Pandas
# ============================================================

# Assicurati che il percorso punti alla cartella corretta del tuo dataset
DATASET_TRAIN_PATH = "/content/drive/MyDrive/progettoLLM/CLARITY/dataset/QEvasion/train"
DATASET_TEST_PATH  = "/content/drive/MyDrive/progettoLLM/CLARITY/dataset/QEvasion/test"

print("Caricamento dataset in corso...")
dataset_train = load_from_disk(DATASET_TRAIN_PATH)
dataset_test  = load_from_disk(DATASET_TEST_PATH)

# Convertiamo in DataFrame Pandas per facilitare l'analisi statistica
df_train = dataset_train.to_pandas()
df_test  = dataset_test.to_pandas()

# Aggiungiamo una colonna per identificare lo split
df_train['split'] = 'Train'
df_test['split'] = 'Test'

# Uniamo tutto in un unico DataFrame generale
df_all = pd.concat([df_train, df_test], ignore_index=True)

print(f"✅ Dataset caricato!")
print(f" - Esempi in Train: {len(df_train)}")
print(f" - Esempi in Test:  {len(df_test)}")
print(f" - Totale Esempi:   {len(df_all)}")

display(df_all.head(3))

In [ ]:
# ============================================================
# CELLA 3 — Distribuzione Macro-Categorie (Task 1: Clarity)
# ============================================================

print("--- Distribuzione Etichette di Chiarezza (Task 1) ---")
clarity_counts = df_all['clarity_label'].value_counts()
print(clarity_counts)
print(f"\nPercentuali:\n{df_all['clarity_label'].value_counts(normalize=True) * 100}")

plt.figure(figsize=(8, 5))
ax = sns.countplot(data=df_all, x='clarity_label', hue='split', palette='Set2', 
                   order=["Clear Reply", "Ambivalent", "Clear Non-Reply"])

plt.title('Distribuzione delle Macro-Categorie di Chiarezza (Train vs Test)', fontsize=14)
plt.xlabel('Categoria (Task 1)', fontsize=12)
plt.ylabel('Numero di Esempi', fontsize=12)

# Salva il grafico
plt.savefig(f"{ANALYSIS_DIR}/distribuzione_clarity.png", bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
# ============================================================
# CELLA 4 — Distribuzione Tecniche di Evasione (Task 2)
# ============================================================

print("--- Distribuzione Tecniche di Evasione (Task 2) ---")
evasion_counts = df_all['evasion_label'].value_counts()
print(evasion_counts)

plt.figure(figsize=(12, 6))
ax = sns.countplot(data=df_all, y='evasion_label', hue='split', palette='viridis', 
                   order=evasion_counts.index)

plt.title('Distribuzione delle Tecniche di Evasione (Train vs Test)', fontsize=14)
plt.xlabel('Numero di Esempi', fontsize=12)
plt.ylabel('Tecnica di Evasione (Task 2)', fontsize=12)
plt.legend(title='Split', loc='lower right')

plt.savefig(f"{ANALYSIS_DIR}/distribuzione_evasion.png", bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
# ============================================================
# CELLA 5 — Analisi della Lunghezza dei Testi (Word Count)
# ============================================================

# Calcoliamo il numero di parole per domanda e risposta
df_all['word_count_question'] = df_all['interview_question'].apply(lambda x: len(str(x).split()))
df_all['word_count_answer']   = df_all['interview_answer'].apply(lambda x: len(str(x).split()))

print("--- Statistiche Lunghezza Testi (Numero di Parole) ---")
display(df_all[['word_count_question', 'word_count_answer']].describe())

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Istogramma Domande
sns.histplot(df_all['word_count_question'], bins=30, kde=True, color='skyblue', ax=axes[0])
axes[0].set_title('Distribuzione Lunghezza Domande')
axes[0].set_xlabel('Numero di parole')
axes[0].set_ylabel('Frequenza')

# Istogramma Risposte
sns.histplot(df_all['word_count_answer'], bins=30, kde=True, color='salmon', ax=axes[1])
axes[1].set_title('Distribuzione Lunghezza Risposte')
axes[1].set_xlabel('Numero di parole')
axes[1].set_ylabel('Frequenza')

plt.tight_layout()
plt.savefig(f"{ANALYSIS_DIR}/lunghezza_testi.png", bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
# ============================================================
# CELLA 6 — Matrice di Correlazione (Task 2 -> Task 1)
# ============================================================
# Vogliamo vedere esattamente come le etichette di Evasione si 
# traducono in etichette di Chiarezza nel Dataset Reale (Ground Truth).

print("--- Crosstab: Evasion Label vs Clarity Label ---")
crosstab_df = pd.crosstab(df_all['evasion_label'], df_all['clarity_label'])

plt.figure(figsize=(10, 8))
sns.heatmap(crosstab_df, annot=True, fmt='d', cmap='YlGnBu', cbar=True)

plt.title('Correlazione nel Dataset: Tecniche di Evasione vs Macro-Chiarezza', fontsize=14)
plt.xlabel('Clarity Label (Task 1)', fontsize=12)
plt.ylabel('Evasion Label (Task 2)', fontsize=12)

plt.savefig(f"{ANALYSIS_DIR}/heatmap_correlazione_t1_t2.png", bbox_inches='tight', dpi=300)
plt.show()

# Salvataggio del dataset con le statistiche calcolate
df_all.to_csv(f"{ANALYSIS_DIR}/dataset_statistiche_complete.csv", index=False)
print(f"Analisi completata! Grafici e dati salvati in: {ANALYSIS_DIR}")